# 州の平均注文額の差をカテゴリ別に分ける

## なぜこの計算をするか

PB・APの平均注文額はSPより高い。州の平均注文額を各カテゴリの金額に分けると、どのカテゴリの過去の売上構成に差があるかを確認できる。

**カテゴリの平均注文額への構成額 = カテゴリGMV ÷ 州のユニークな注文数**。全カテゴリの構成額を足すと、その州の平均注文額になる。PB・APの構成額からSPの構成額を引いた差を並べる。

この差は観察された注文構成の差であり、カテゴリが注文額を増やした因果効果や販促時の増分GMVではない。少数の高額注文にも注意する。

In [8]:
import pandas as pd
from IPython.display import display

# 以前のNotebookと同じAthena集計結果を読み込む。
df_baseline = pd.read_csv("../data/sales_baseline_by_month_state.csv")
df_category = pd.read_csv("../data/sales_by_category_state.csv")

# カテゴリ別注文数には重複があるため、州の注文数は月×州の集計から求める。
state_totals = (
    df_baseline.groupby("customer_state", as_index=False)
    .agg(state_orders=("order_count", "sum"), state_gmv=("gmv", "sum"))
)
state_totals["average_order_value"] = (
    state_totals["state_gmv"] / state_totals["state_orders"]
)

# 各カテゴリが州の平均注文額を何単位分構成するかを計算する。
category_parts = df_category.merge(
    state_totals[["customer_state", "state_orders"]],
    on="customer_state",
    how="left",
)
category_parts["aov_part"] = (
    category_parts["category_gmv"] / category_parts["state_orders"]
)

display(
    state_totals.loc[
        state_totals["customer_state"].isin(["SP", "PB", "AP"]),
        ["customer_state", "state_orders", "average_order_value"],
    ].round(2)
)

,customer_state,state_orders,average_order_value
3,AP,67,199.62
14,PB,517,217.77
25,SP,40501,125.12


In [9]:
# 州×カテゴリを横持ちにし、存在しない組み合わせは0とする。
parts = category_parts.pivot(
    index="product_category", columns="customer_state", values="aov_part"
).fillna(0)
counts = category_parts.pivot(
    index="product_category",
    columns="customer_state",
    values="category_order_count",
).fillna(0).astype(int)

for state in ["PB", "AP"]:
    # 各カテゴリの構成額をSPと比べる。差の合計が州間の平均注文額差になる。
    comparison = pd.DataFrame({
        "product_category": parts.index,
        f"{state}_category_orders": counts[state].to_numpy(),
        "SP_category_orders": counts["SP"].to_numpy(),
        f"{state}_aov_part": parts[state].to_numpy(),
        "SP_aov_part": parts["SP"].to_numpy(),
    })
    comparison["aov_part_difference"] = (
        comparison[f"{state}_aov_part"] - comparison["SP_aov_part"]
    )

    print(f"{state}とSPの平均注文額差: {comparison['aov_part_difference'].sum():.2f}")
    display(comparison.nlargest(10, "aov_part_difference").round(2))

PBとSPの平均注文額差: 92.65


,product_category,PB_category_orders,SP_category_orders,PB_aov_part,SP_aov_part,aov_part_difference
43,health_beauty,75,3715,30.35,11.21,19.14
73,watches_gifts,41,2090,25.34,10.41,14.94
14,computers,4,50,13.79,1.53,12.26
42,garden_tools,14,1242,10.81,3.66,7.15
50,industry_commerce_and_business,3,115,6.43,0.46,5.97
15,computers_accessories,40,2616,13.92,8.42,5.50
20,cool_stuff,22,1289,10.56,5.15,5.41
72,uncategorized,8,562,5.31,1.55,3.75
26,electronics,13,928,4.98,1.27,3.70
70,telephony,24,1491,6.09,2.45,3.64


APとSPの平均注文額差: 74.50


,product_category,AP_category_orders,SP_category_orders,AP_aov_part,SP_aov_part,aov_part_difference
15,computers_accessories,6,2616,30.60,8.42,22.18
14,computers,1,50,21.45,1.53,19.92
73,watches_gifts,7,2090,24.47,10.41,14.07
5,auto,3,1585,18.16,5.17,12.99
43,health_beauty,10,3715,20.61,11.21,9.40
0,agro_industry_and_commerce,1,84,8.81,0.64,8.17
26,electronics,3,928,9.07,1.27,7.79
34,fixed_telephony,2,95,3.48,0.32,3.16
67,sports_leisure,6,3216,12.13,9.26,2.87
56,musical_instruments,2,247,4.39,1.67,2.72


## 結果をどう読むか

- 差が大きいカテゴリでは、対象州の件数も確認する。APのように州全体の注文数が少ない場合、数件で構成額が大きく変わる。
- この表は州間の平均注文額差を分けたもの。購入割合の差とカテゴリ内の金額差をさらに区別するには、追加の計算が必要。
- 販促対象の選定には、期間ごとの安定性、商品・販売者の供給、広告費と利益率なども必要。次の問いは表を見てから選ぶ。